In [31]:
from google.colab import drive
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/3CSE Machine Learning Group 4/'
scaler_path = base_path + 'FINAL PROJECT/Dataset/Model_Ready_Exports/feature_scaler.pkl'
x_test_path = base_path + 'FINAL PROJECT/Dataset/Model_Ready_Exports/X_test.parquet'
y_test_path = base_path + 'FINAL PROJECT/Dataset/Model_Ready_Exports/y_test.parquet'
xgb_model_path = base_path + 'FINAL PROJECT/XGB/xgb_json/xgboost_booster.json' #Modified to new path json

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import joblib
import xgboost as xgb
from sklearn.linear_model import LinearRegression

In [33]:
# load data and scaler
scaler = joblib.load(scaler_path)
X_test = pd.read_parquet(x_test_path)
y_test = pd.read_parquet(y_test_path).squeeze()

In [34]:
# reconstruct MLR model (coefficients from Vince’s notebook)
mlr = LinearRegression()
mlr.intercept_ = 12.0586

mlr.coef_ = np.array([
    0.205389,   # rent
    0.006595,   # for_sale
   -0.043229,   # days_pending
    0.027659,   # price_cuts
    0.088828,   # market_heat_index
    0.022907,   # new_homeowner_affordability
   -0.076276,   # space_premium
    0.025667,   # is_post_pandemic
    0.084938    # state_encoded
])

In [35]:
# load XGBoost booster
xgb_booster = xgb.Booster()
xgb_booster.load_model(xgb_model_path)
dtest = xgb.DMatrix(X_test.values, feature_names=X_test.columns.tolist())

In [36]:
# sample for SHAP (2000 rows)
np.random.seed(42)
sample_idx = np.random.choice(len(X_test), size=min(2000, len(X_test)), replace=False)
X_sample = X_test.iloc[sample_idx]
dtest_sample = xgb.DMatrix(X_sample.values, feature_names=X_test.columns.tolist())

In [37]:
# MLR SHAP (LinearExplainer)
print("Running LinearExplainer...")
explainer_linear = shap.LinearExplainer(mlr, X_test, feature_perturbation="interventional")
shap_values_linear = explainer_linear.shap_values(X_sample)

# Bar plot
plt.figure()
shap.summary_plot(shap_values_linear, X_sample, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig('shap_mlr_bar.png', dpi=300)
plt.close()

# Beeswarm
plt.figure()
shap.summary_plot(shap_values_linear, X_sample, show=False)
plt.tight_layout()
plt.savefig('shap_mlr_beeswarm.png', dpi=300)
plt.close()

Running LinearExplainer...


/usr/local/lib/python3.12/dist-packages/shap/explainers/_linear.py:123: FutureWarning: The feature_perturbation option is now deprecated in favor of using the appropriate masker (maskers.Independent, maskers.Partition or maskers.Impute).
  warnings.warn(wmsg, FutureWarning)
/tmp/ipykernel_1728/3660356659.py:8: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values_linear, X_sample, plot_type="bar", show=False)
/tmp/ipykernel_1728/3660356659.py:15: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values_linear, X_sample, show=False)


In [38]:
# Check what features the XGBoost model expects
print("Booster feature names:", xgb_booster.feature_names)
print("X_test columns:", X_test.columns.tolist())
print("Booster num features:", getattr(xgb_booster, 'num_features', 'unknown'))

Booster feature names: ['rent', 'for_sale', 'days_pending', 'price_cuts', 'market_heat_index', 'new_homeowner_affordability', 'space_premium', 'is_post_pandemic', 'state_encoded', 'Time_Index']
X_test columns: ['rent', 'for_sale', 'days_pending', 'price_cuts', 'market_heat_index', 'new_homeowner_affordability', 'space_premium', 'is_post_pandemic', 'state_encoded']
Booster num features: <bound method Booster.num_features of <xgboost.core.Booster object at 0x78bdff969fd0>>


In [39]:
start_idx = int(112455 - len(X_test))   # should be 98532
X_test['Time_Index'] = np.arange(start_idx, start_idx + len(X_test))

# Ensure the columns are in the exact order the booster expects
expected_cols = ['rent', 'for_sale', 'days_pending', 'price_cuts', 'market_heat_index',
                 'new_homeowner_affordability', 'space_premium', 'is_post_pandemic',
                 'state_encoded', 'Time_Index']
X_test = X_test[expected_cols]

# Recreate the DMatrix and the sample
dtest = xgb.DMatrix(X_test.values, feature_names=expected_cols)
X_sample = X_test.iloc[sample_idx]
dtest_sample = xgb.DMatrix(X_sample.values, feature_names=expected_cols)

print("Time_Index added. X_test now has", X_test.shape[1], "features.")

Time_Index added. X_test now has 10 features.


In [40]:
# XGBoost SHAP (TreeExplainer)
print("Running TreeExplainer...")
explainer_tree = shap.TreeExplainer(xgb_booster)
shap_values_tree = explainer_tree.shap_values(dtest_sample)

# Bar plot
plt.figure()
shap.summary_plot(shap_values_tree, X_sample, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig('shap_xgb_bar.png', dpi=300)
plt.close()

# Beeswarm
plt.figure()
shap.summary_plot(shap_values_tree, X_sample, show=False)
plt.tight_layout()
plt.savefig('shap_xgb_beeswarm.png', dpi=300)
plt.close()

# Dependence plots for top 5 features
mean_abs_shap = np.abs(shap_values_tree).mean(axis=0)
top_indices = np.argsort(mean_abs_shap)[::-1][:5]

for feat_idx in top_indices:
    feat_name = X_test.columns[feat_idx]
    plt.figure()
    shap.dependence_plot(
        ind=feat_idx,
        shap_values=shap_values_tree,
        features=X_sample,
        feature_names=X_test.columns.tolist(),
        show=False
    )
    plt.tight_layout()
    plt.savefig(f'shap_xgb_dependence_{feat_name}.png', dpi=300)
    plt.close()

print("All SHAP plots saved to the current Colab directory.")

Running TreeExplainer...


/tmp/ipykernel_1728/3187307280.py:8: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values_tree, X_sample, plot_type="bar", show=False)
/tmp/ipykernel_1728/3187307280.py:15: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values_tree, X_sample, show=False)


All SHAP plots saved to the current Colab directory.


<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

In [41]:
import shutil, glob
dest = base_path + 'FINAL PROJECT/XGB/'
shutil.copy('shap_mlr_bar.png', dest)
shutil.copy('shap_mlr_beeswarm.png', dest)
shutil.copy('shap_xgb_bar.png', dest)
shutil.copy('shap_xgb_beeswarm.png', dest)
for f in glob.glob('shap_xgb_dependence_*.png'):
    shutil.copy(f, dest)